# Compensação e perfil dos municípios que mudam de posição (PS2 e PS4)

Este notebook responde a duas perguntas secundárias do guia de escrita:

- **PS2 (compensação):** o grau de compensação do método explica as divergências entre os rankings e as diferenças de associação com a DRSAI? Os municípios com desempenho desigual entre os indicadores, como os que têm zero em Tratamento ou em Destinação, são os que mais mudam?
- **PS4 (perfil):** qual perfil caracteriza os municípios que mudam de quartil, e o que isso sugere para a priorização?

Entrada: `planilhas/Relatorio ABES 2026 NE.ods`. Os 16 rankings são recalculados com `mcda.py`, como nos notebooks 01 e 02. O notebook não grava arquivos. A DRSAI não altera nenhum ranking; ela só aparece nas seções 4 e 5, como no protocolo do log, e na seção 7, uma análise exploratória acrescentada depois do teste (seção 10 do log).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import mcda

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir())
SEMENTE = 20261109
ABES = "Soma ponderada | Pesos iguais"
MAIS_DRSAI = "VIKOR | Pesos iguais"  # maior |ρ| com a DRSAI no notebook 02

base = mcda.ler_base(RAIZ)
X = base[mcda.CRITERIOS].to_numpy(float)
pont = mcda.rankings(X)
pos = pont.apply(mcda.posicoes)
ORDEM = [f"{m} | {e}" for m in mcda.METODOS for e in mcda.ESQUEMAS]
pd.set_option("display.max_columns", 30)

## 1. Perfil de desempenho de cada município

- **Desigualdade:** desvio-padrão dos cinco indicadores (notas brutas, 0 a 100).
- **Pior indicador:** o menor dos cinco.
- **Zeros:** quantos dos cinco valem 0. Em Tratamento e Destinação, os zeros são frequentes.

In [2]:
perfil = base[["Município", "UF", "Porte", "Categoria"]].copy()
for c, nome in zip(mcda.CRITERIOS, mcda.NOMES):
    perfil[nome] = base[c]
perfil["Média"] = X.mean(axis=1)
perfil["Desigualdade (DP)"] = X.std(axis=1)
perfil["Pior indicador"] = X.min(axis=1)
perfil["Zeros"] = (X == 0).sum(axis=1)
perfil["DRSAI"] = base["Taxa de Internações por DRSAI"].where(base["Taxa de Internações por DRSAI"] > 0)
print("Municípios por número de zeros:"); display(perfil["Zeros"].value_counts().sort_index().to_frame("municípios").T)
print("Zeros por indicador:"); display((perfil[mcda.NOMES] == 0).sum().to_frame("municípios com 0").T)
perfil[["Média", "Desigualdade (DP)", "Pior indicador"]].describe().round(1)

Municípios por número de zeros:


Zeros,0,1,2
municípios,165,276,64


Zeros por indicador:


,Água,Esgoto,Tratamento,Coleta,Destinação
municípios com 0,1,1,154,0,248


,Média,Desigualdade (DP),Pior indicador
count,505.0,505.0,505.0
mean,52.1,32.7,9.1
std,15.8,7.8,18.4
min,19.7,4.2,0.0
25%,39.5,29.0,0.0
50%,51.4,33.9,0.0
75%,63.0,38.2,8.5
max,96.6,46.3,88.6


## 2. Quanto o método e os pesos movem cada município

Para cada município, a amplitude de posições (maior − menor) entre os quatro métodos com os mesmos pesos, em média sobre os quatro esquemas, mede o efeito do método. A amplitude entre os quatro esquemas com o mesmo método, em média sobre os métodos, mede o efeito dos pesos.

In [3]:
def amplitude(cols):
    d = pos[cols]
    return d.max(axis=1) - d.min(axis=1)

ef_metodo = np.mean([amplitude([f"{m} | {e}" for m in mcda.METODOS]) for e in mcda.ESQUEMAS], axis=0)
ef_pesos = np.mean([amplitude([f"{m} | {e}" for e in mcda.ESQUEMAS]) for m in mcda.METODOS], axis=0)
perfil["Amplitude: método"] = ef_metodo
perfil["Amplitude: pesos"] = ef_pesos
perfil["Amplitude: 16 rankings"] = amplitude(ORDEM)

print(f"Amplitude média devida ao método: {ef_metodo.mean():.0f} posições; devida aos pesos: {ef_pesos.mean():.0f} posições")
assoc = pd.DataFrame({
    v: [stats.spearmanr(perfil[v], perfil[a])[0] for a in ["Amplitude: método", "Amplitude: pesos", "Amplitude: 16 rankings"]]
    for v in ["Desigualdade (DP)", "Zeros", "Pior indicador", "Média"]
}, index=["Amplitude: método", "Amplitude: pesos", "Amplitude: 16 rankings"])
print("Spearman ρ entre a amplitude de posições e o perfil do município")
assoc.round(3)

Amplitude média devida ao método: 62 posições; devida aos pesos: 132 posições
Spearman ρ entre a amplitude de posições e o perfil do município


,Desigualdade (DP),Zeros,Pior indicador,Média
Amplitude: método,0.268,-0.099,-0.031,0.068
Amplitude: pesos,0.307,0.197,-0.283,-0.163
Amplitude: 16 rankings,0.307,0.177,-0.238,-0.151


## 3. Em que direção cada método move os municípios desiguais

Δposição = posição no método − posição na soma ponderada com os mesmos pesos. **Δ positivo é queda** (o município fica pior colocado que na soma ponderada). Se o método penaliza quem tem um indicador muito fraco, Δ cresce com o número de zeros.

In [4]:
linhas = []
for e in mcda.ESQUEMAS:
    for m in ["TOPSIS", "VIKOR", "PROMETHEE II"]:
        d = pos[f"{m} | {e}"] - pos[f"Soma ponderada | {e}"]
        linhas.append({
            "Pesos": e, "Método": m,
            "ρ(Δ, zeros)": stats.spearmanr(d, perfil["Zeros"])[0],
            "ρ(Δ, desigualdade)": stats.spearmanr(d, perfil["Desigualdade (DP)"])[0],
            "Δ médio, 0 em Tratamento": d[perfil["Tratamento"] == 0].mean(),
            "Δ médio, 0 em Destinação": d[perfil["Destinação"] == 0].mean(),
            "Δ médio, sem zeros": d[perfil["Zeros"] == 0].mean(),
        })
direcao = pd.DataFrame(linhas).set_index(["Método", "Pesos"]).sort_index()
direcao.round(2)

ρ(Δ, zeros)  ρ(Δ, desigualdade)  \
Método       Pesos                                            
PROMETHEE II AHP Gaussiano        -0.11               -0.45   
             AHP clássico         -0.20               -0.60   
             CRITIC               -0.29               -0.20   
             Pesos iguais         -0.45               -0.21   
TOPSIS       AHP Gaussiano        -0.50                0.16   
             AHP clássico         -0.52               -0.18   
             CRITIC               -0.07                0.60   
             Pesos iguais         -0.40                0.45   
VIKOR        AHP Gaussiano         0.56               -0.13   
             AHP clássico         -0.22               -0.43   
             CRITIC                0.25                0.27   
             Pesos iguais          0.43                0.04   

                            Δ médio, 0 em Tratamento  \
Método       Pesos                                     
PROMETHEE II AHP Gaussiano                    -20.31   
             AHP clássico                       0.15   
             CRITIC                             7.21   
             Pesos iguais                      10.05   
TOPSIS       AHP Gaussiano                     -2.49   
             AHP clássico                      -5.48   
             CRITIC                             1.92   
             Pesos iguais                       7.84   
VIKOR        AHP Gaussiano                     25.93   
             AHP clássico                       1.26   
             CRITIC                             8.77   
             Pesos iguais                      17.49   

                            Δ médio, 0 em Destinação  Δ médio, sem zeros  
Método       Pesos                                                        
PROMETHEE II AHP Gaussiano                      7.47                0.63  
             AHP clássico                     -18.31               14.97  
             CRITIC                           -12.85                9.48  
             Pesos iguais                     -31.58               25.58  
TOPSIS       AHP Gaussiano                    -10.12               11.84  
             AHP clássico                     -29.81               29.61  
             CRITIC                            -4.31                2.06  
             Pesos iguais                     -32.17               32.74  
VIKOR        AHP Gaussiano                     10.05              -29.02  
             AHP clássico                     -18.56               13.10  
             CRITIC                             3.08              -12.58  
             Pesos iguais                       6.24              -24.83

In [5]:
print("Média sobre os quatro esquemas de pesos")
direcao.groupby(level="Método").mean().loc[["TOPSIS", "VIKOR", "PROMETHEE II"]].round(2)

Média sobre os quatro esquemas de pesos


,"ρ(Δ, zeros)","ρ(Δ, desigualdade)","Δ médio, 0 em Tratamento","Δ médio, 0 em Destinação","Δ médio, sem zeros"
Método,,,,,
TOPSIS,-0.37,0.26,0.44,-19.10,19.06
VIKOR,0.26,-0.06,13.36,0.20,-13.33
PROMETHEE II,-0.26,-0.37,-0.72,-13.82,12.66


## 4. O componente mais fraco e a DRSAI

A hipótese do guia é que o risco sanitário depende do componente mais fraco. Antes de olhar os métodos, ela pode ser testada no próprio perfil: a taxa de DRSAI se associa mais ao pior indicador do município do que à média dos cinco? Spearman nos 360 municípios com taxa válida, com intervalo bootstrap da diferença (5.000 reamostragens, semente do log).

In [6]:
val = perfil[perfil["DRSAI"].notna()].reset_index(drop=True)
y = val["DRSAI"].to_numpy()
variaveis = ["Média", "Pior indicador", "Zeros", "Desigualdade (DP)"] + mcda.NOMES
rho_perfil = pd.Series({v: stats.spearmanr(val[v], y)[0] for v in variaveis}, name="ρ com a DRSAI")

rng = np.random.default_rng(SEMENTE)
idx = rng.integers(0, len(y), size=(5000, len(y)))
def rho(a, b): return stats.spearmanr(a, b)[0]
d_min = np.array([rho(val["Pior indicador"].to_numpy()[i], y[i]) - rho(val["Média"].to_numpy()[i], y[i]) for i in idx])
d_zer = np.array([-rho(val["Zeros"].to_numpy()[i], y[i]) - rho(val["Média"].to_numpy()[i], y[i]) for i in idx])
display(rho_perfil.round(3).to_frame().T)
print(f"ρ(pior) − ρ(média) = {rho_perfil['Pior indicador'] - rho_perfil['Média']:+.3f}, IC 95% [{np.percentile(d_min, 2.5):+.3f}; {np.percentile(d_min, 97.5):+.3f}]")
print(f"−ρ(zeros) − ρ(média) = {-rho_perfil['Zeros'] - rho_perfil['Média']:+.3f}, IC 95% [{np.percentile(d_zer, 2.5):+.3f}; {np.percentile(d_zer, 97.5):+.3f}]  (valor negativo: os zeros se associam mais que a média)")

,Média,Pior indicador,Zeros,Desigualdade (DP),Água,Esgoto,Tratamento,Coleta,Destinação
ρ com a DRSAI,-0.218,-0.221,0.247,0.014,-0.125,0.024,-0.104,-0.125,-0.223


ρ(pior) − ρ(média) = -0.003, IC 95% [-0.084; +0.077]
−ρ(zeros) − ρ(média) = -0.029, IC 95% [-0.112; +0.055]  (valor negativo: os zeros se associam mais que a média)


**Compensação e associação com a DRSAI.** O ρ médio de cada método com a DRSAI (do notebook 02), ao lado de quanto o método penaliza os zeros (seção 3). Se a hipótese do componente mais fraco valer, os métodos que mais penalizam os zeros deveriam ter o ρ mais negativo.

In [7]:
rho_rk = pd.Series({c: stats.spearmanr(pont.loc[perfil["DRSAI"].notna(), c], perfil["DRSAI"].dropna())[0] for c in ORDEM})
comp = pd.DataFrame({
    "Compensação": ["Total", "Total", "Parcial", "Limitada"],
    "ρ médio com a DRSAI": [rho_rk[[f"{m} | {e}" for e in mcda.ESQUEMAS]].mean() for m in mcda.METODOS],
    "ρ(Δ posição, zeros) médio": [0] + list(direcao.groupby(level="Método")["ρ(Δ, zeros)"].mean().loc[["TOPSIS", "VIKOR", "PROMETHEE II"]]),
}, index=mcda.METODOS)
comp.round(3)

,Compensação,ρ médio com a DRSAI,"ρ(Δ posição, zeros) médio"
Soma ponderada,Total,-0.191,0.000
TOPSIS,Total,-0.149,-0.371
VIKOR,Parcial,-0.204,0.256
PROMETHEE II,Limitada,-0.185,-0.260


## 5. Perfil de quem muda de quartil: o ranking mais associado à DRSAI contra o ABES

O ranking com o ρ mais negativo no notebook 02 é o VIKOR com pesos iguais. Aqui ele é comparado com o ABES nos 505 municípios. Quartil 1 = 25% melhores. O **indicador decisivo** de um município no VIKOR é o que define o seu arrependimento R, isto é, o indicador com a maior distância ponderada ao melhor valor; é por ele que o VIKOR se afasta da soma.

In [8]:
def quartil(c):
    return pd.qcut(pos[c].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)

perfil["Posição ABES"] = pos[ABES]
perfil["Posição VIKOR"] = pos[MAIS_DRSAI]
perfil["Quartil ABES"] = quartil(ABES)
perfil["Quartil VIKOR"] = quartil(MAIS_DRSAI)
perfil["Δ quartil"] = perfil["Quartil VIKOR"] - perfil["Quartil ABES"]
d = (X.max(axis=0) - X) / (X.max(axis=0) - X.min(axis=0))
perfil["Indicador decisivo"] = np.array(mcda.NOMES)[(0.2 * d).argmax(axis=1)]

print("Transição de quartis (linhas: ABES; colunas: VIKOR com pesos iguais)")
display(pd.crosstab(perfil["Quartil ABES"], perfil["Quartil VIKOR"], margins=True, margins_name="Total"))
print("Categoria do ABES × quartil no VIKOR")
pd.crosstab(perfil["Categoria"], perfil["Quartil VIKOR"])

Transição de quartis (linhas: ABES; colunas: VIKOR com pesos iguais)


Quartil VIKOR,1,2,3,4,Total
Quartil ABES,,,,,
1,101,26,0,0,127
2,26,91,9,0,126
3,0,8,113,5,126
4,0,1,4,121,126
Total,127,126,126,126,505


Categoria do ABES × quartil no VIKOR


Quartil VIKOR,1,2,3,4
Categoria,,,,
Compromisso com a universalização,7,0,0,0
Empenho para universalização,120,125,120,2
Primeiros passos para a universalização,0,1,6,124


In [9]:
grupo = np.select([perfil["Δ quartil"] < 0, perfil["Δ quartil"] > 0], ["Sobe", "Desce"], "Mantém")
perfil["Grupo"] = grupo
resumo = perfil.groupby("Grupo").agg(
    municípios=("Município", "size"),
    média=("Média", "mean"),
    desigualdade=("Desigualdade (DP)", "mean"),
    pior_indicador=("Pior indicador", "mean"),
    zeros=("Zeros", "mean"),
    zero_tratamento=("Tratamento", lambda s: (s == 0).mean() * 100),
    zero_destinação=("Destinação", lambda s: (s == 0).mean() * 100),
    grande_porte=("Porte", lambda s: (s == "Grande porte").mean() * 100),
    DRSAI_mediana=("DRSAI", "median"),
).loc[["Sobe", "Mantém", "Desce"]]
print("Perfil médio (zero_tratamento, zero_destinação e grande_porte em %)")
resumo.round(1)

Perfil médio (zero_tratamento, zero_destinação e grande_porte em %)


,municípios,média,desigualdade,pior_indicador,zeros,zero_tratamento,zero_destinação,grande_porte,DRSAI_mediana
Grupo,,,,,,,,,
Sobe,39,54.2,31.9,17.5,0.1,0.0,10.3,12.8,59.4
Mantém,426,51.3,32.6,9.2,0.8,31.9,52.6,12.4,52.5
Desce,40,59.1,34.4,0.2,1.0,45.0,50.0,5.0,38.5


In [10]:
print("Indicador decisivo dos que mudam de quartil")
pd.crosstab(perfil.loc[perfil["Grupo"] != "Mantém", "Indicador decisivo"], perfil.loc[perfil["Grupo"] != "Mantém", "Grupo"])

Indicador decisivo dos que mudam de quartil


Grupo,Desce,Sobe
Indicador decisivo,,
Coleta,0,4
Destinação,19,4
Esgoto,0,17
Tratamento,21,14


**Os municípios que mudam de quartil**, do que mais sobe ao que mais desce.

In [11]:
cols = ["Município", "UF", "Porte", "Posição ABES", "Posição VIKOR", "Quartil ABES", "Quartil VIKOR"] + mcda.NOMES + ["Indicador decisivo", "DRSAI"]
mudam = perfil[perfil["Grupo"] != "Mantém"].assign(Δ=lambda t: t["Posição VIKOR"] - t["Posição ABES"]).sort_values("Δ")
mudam[cols + ["Δ"]].round(1).reset_index(drop=True)

,Município,UF,Porte,Posição ABES,Posição VIKOR,Quartil ABES,Quartil VIKOR,Água,Esgoto,Tratamento,Coleta,Destinação,Indicador decisivo,DRSAI,Δ
0,Dormentes,PE,Pequeno e médio porte,393,196,4,2,41.0,24.6,27.5,45.4,53.4,Esgoto,62.0,-197
1,Quixelô,CE,Pequeno e médio porte,370,236,3,2,42.2,33.4,41.0,33.4,53.0,Coleta,100.0,-134
2,Orobó,PE,Pequeno e médio porte,218,88,2,1,37.0,35.5,28.1,71.3,100.0,Tratamento,4.5,-130
3,Quixaba,PB,Pequeno e médio porte,209,81,2,1,36.1,53.3,33.2,53.3,100.0,Tratamento,NaN,-128
4,Tacaimbó,PE,Pequeno e médio porte,208,82,2,1,62.5,35.6,34.1,44.8,100.0,Coleta,NaN,-126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,Juazeiro,BA,Grande porte,89,140,1,2,94.4,83.9,77.0,79.6,0.0,Destinação,27.1,51
75,Porto Alegre do Piauí,PI,Pequeno e médio porte,112,163,1,2,78.8,55.2,100.0,86.8,0.0,Destinação,NaN,51
76,Serra Negra do Norte,RN,Pequeno e médio porte,104,156,1,2,73.1,70.0,100.0,83.3,0.0,Destinação,38.5,52
77,Tenente Ananias,RN,Pequeno e médio porte,118,172,1,2,79.0,67.0,0.0,72.3,100.0,Tratamento,2238.6,54


## 6. Os que mais mudam nos 16 rankings

Os 20 municípios com a maior amplitude de posições entre os 16 rankings, com o ranking em que ficam melhor e pior colocados.

In [12]:
top = perfil.nlargest(20, "Amplitude: 16 rankings").copy()
top["Melhor posição"] = pos.loc[top.index, ORDEM].min(axis=1)
top["Melhor ranking"] = pos.loc[top.index, ORDEM].idxmin(axis=1)
top["Pior posição"] = pos.loc[top.index, ORDEM].max(axis=1)
top["Pior ranking"] = pos.loc[top.index, ORDEM].idxmax(axis=1)
top[["Município", "UF"] + mcda.NOMES + ["Posição ABES", "Melhor posição", "Melhor ranking", "Pior posição", "Pior ranking"]].round(1).reset_index(drop=True)

,Município,UF,Água,Esgoto,Tratamento,Coleta,Destinação,Posição ABES,Melhor posição,Melhor ranking,Pior posição,Pior ranking
0,Carnaubeira da Penha,PE,17.0,14.2,100.0,30.9,100.0,240,29,Soma ponderada | AHP Gaussiano,500,VIKOR | AHP clássico
1,Santa Filomena,PE,36.7,12.1,61.1,36.7,100.0,271,59,VIKOR | AHP Gaussiano,493,PROMETHEE II | AHP clássico
2,Guamaré,RN,27.0,55.2,100.0,94.1,100.0,50,11,TOPSIS | AHP Gaussiano,433,VIKOR | AHP clássico
3,Santa Luzia do Itanhy,SE,25.1,15.2,32.6,57.7,100.0,303,87,VIKOR | AHP Gaussiano,500,TOPSIS | AHP clássico
4,Quixelô,CE,42.2,33.4,41.0,33.4,53.0,370,86,VIKOR | AHP Gaussiano,488,PROMETHEE II | AHP clássico
5,Quixaba,PB,36.1,53.3,33.2,53.3,100.0,209,70,TOPSIS | CRITIC,468,VIKOR | AHP clássico
6,Mamanguape,PB,33.4,12.9,34.5,87.9,100.0,229,76,VIKOR | AHP Gaussiano,471,TOPSIS | AHP clássico
7,Saloá,PE,37.9,46.9,100.0,73.2,96.8,69,16,Soma ponderada | AHP Gaussiano,409,VIKOR | AHP clássico
8,Pacoti,CE,27.1,25.6,71.4,100.0,0.0,326,87,TOPSIS | AHP Gaussiano,473,VIKOR | AHP clássico
9,São Gonçalo do Amarante,CE,36.8,22.8,100.0,90.4,0.0,262,54,TOPSIS | AHP Gaussiano,437,VIKOR | AHP clássico


In [13]:
print("Quantos dos 20 ficam pior colocados com pesos de cada esquema")
display(top["Pior ranking"].str.split(" | ", regex=False).str[1].value_counts().to_frame("municípios").T)
print("e melhor colocados")
top["Melhor ranking"].str.split(" | ", regex=False).str[1].value_counts().to_frame("municípios").T

Quantos dos 20 ficam pior colocados com pesos de cada esquema


Pior ranking,AHP clássico
municípios,20


e melhor colocados


Melhor ranking,AHP Gaussiano,CRITIC
municípios,19,1


## 7. Lista de prioridade de consenso (análise exploratória)

**Esta seção foi acrescentada depois do teste com a DRSAI** e está registrada na seção 10 do `docs/Log de decisoes.md`. Ela não altera nenhum ranking nem parâmetro; só combina os 16 rankings prontos para indicar prioridades. Os métodos multicritério apoiam a decisão, e não a substituem: em vez de escolher um único ranking, a lista reúne os municípios que ficam mal colocados em todas as combinações aceitáveis.

1. **Filtro pela DRSAI.** Saem os rankings que se associam significativamente menos à DRSAI que o ABES (IC de 95% do Δρ contra o ABES acima de zero), com as mesmas 5.000 reamostragens e a mesma semente do notebook 02. Ficam 9 rankings.
2. **Saneamento como direito.** Entre os 9, ficam os métodos de compensação parcial ou limitada (VIKOR e PROMETHEE II), em que um bom indicador não compensa por inteiro a falta de outro. Ficam 6 rankings.
3. **Consenso.** Prioritários são os municípios no 4º quartil (25% piores) nos 6 rankings. Os que estão no 4º quartil em alguns, mas não em todos, dependem da escolha de pesos e ficam para análise caso a caso.

In [14]:
N_BOOT = 5000
val = perfil["DRSAI"].notna().to_numpy()
PONT = pont.loc[val, ORDEM].to_numpy()
y = perfil.loc[val, "DRSAI"].to_numpy()
n = len(y)

def spearman_cols(M, v):
    R = np.apply_along_axis(stats.rankdata, 0, M)
    r = stats.rankdata(v)
    R = (R - R.mean(0)) / R.std(0)
    r = (r - r.mean()) / r.std()
    return (R * r[:, None]).mean(0)

rho = spearman_cols(PONT, y)
idx = np.random.default_rng(SEMENTE).integers(0, n, size=(N_BOOT, n))  # as mesmas reamostragens do notebook 02
boot = np.array([spearman_cols(PONT[i], y[i]) for i in idx])
a = ORDEM.index(ABES)
d = boot - boot[:, [a]]
filtro = pd.DataFrame({"ρ": rho, "Δρ contra o ABES": rho - rho[a],
                       "IC 2,5%": np.percentile(d, 2.5, axis=0), "IC 97,5%": np.percentile(d, 97.5, axis=0)}, index=ORDEM)
filtro["Significativamente pior"] = filtro["IC 2,5%"] > 0
filtro["Passa no filtro"] = ~filtro["Significativamente pior"]
filtro["Compensação parcial ou limitada"] = filtro.index.str.startswith(("VIKOR", "PROMETHEE II"))
PASSA = filtro.index[filtro["Passa no filtro"]].tolist()
DIREITO = filtro.index[filtro["Passa no filtro"] & filtro["Compensação parcial ou limitada"]].tolist()
print(len(PASSA), "rankings passam no filtro;", len(DIREITO), "deles têm compensação parcial ou limitada")
filtro.round(4)

9 rankings passam no filtro; 6 deles têm compensação parcial ou limitada


,ρ,Δρ contra o ABES,"IC 2,5%","IC 97,5%",Significativamente pior,Passa no filtro,Compensação parcial ou limitada
Soma ponderada | Pesos iguais,-0.2179,0.0000,0.0000,0.0000,False,True,False
Soma ponderada | CRITIC,-0.2269,-0.0091,-0.0343,0.0152,False,True,False
Soma ponderada | AHP Gaussiano,-0.1679,0.0499,0.0040,0.0969,True,False,False
Soma ponderada | AHP clássico,-0.1509,0.0670,0.0203,0.1164,True,False,False
TOPSIS | Pesos iguais,-0.1370,0.0808,0.0400,0.1232,True,False,False
TOPSIS | CRITIC,-0.2058,0.0121,-0.0207,0.0455,False,True,False
TOPSIS | AHP Gaussiano,-0.1411,0.0768,0.0203,0.1358,True,False,False
TOPSIS | AHP clássico,-0.1135,0.1043,0.0305,0.1773,True,False,False
VIKOR | Pesos iguais,-0.2296,-0.0117,-0.0323,0.0079,False,True,True
VIKOR | CRITIC,-0.2278,-0.0099,-0.0423,0.0224,False,True,True


In [15]:
q4 = pd.DataFrame({c: quartil(c) == 4 for c in ORDEM})
perfil["No 4º quartil (de 6)"] = q4[DIREITO].sum(axis=1)
perfil["No 4º quartil (de 9)"] = q4[PASSA].sum(axis=1)
prior = perfil["No 4º quartil (de 6)"] == len(DIREITO)
print(prior.sum(), "municípios no 4º quartil nos 6 rankings;",
      ((perfil["No 4º quartil (de 6)"] > 0) & ~prior).sum(), "em alguns, mas não em todos")
print((perfil["No 4º quartil (de 9)"] == len(PASSA)).sum(), "no 4º quartil nos 9 rankings que passam no filtro,",
      ((perfil["No 4º quartil (de 9)"] == len(PASSA)) & prior).sum(), "deles também na lista dos 6")

grupos = np.where(prior, "Prioritários", "Demais")
comparacao = perfil.assign(Grupo=grupos).groupby("Grupo").agg(
    municípios=("Município", "size"), **{n: (n, "mean") for n in mcda.NOMES},
    zero_em_Tratamento=("Tratamento", lambda s: (s == 0).mean()),
    zero_em_Destinação=("Destinação", lambda s: (s == 0).mean()),
    DRSAI_válidas=("DRSAI", "count"), DRSAI_mediana=("DRSAI", "median"))
display(comparacao.round(2))
uf = pd.DataFrame({"Prioritários": perfil.loc[prior, "UF"].value_counts(), "Todos os 505": perfil["UF"].value_counts()}).fillna(0).astype(int)
uf["% dos prioritários"] = (100 * uf["Prioritários"] / prior.sum()).round(1)
uf["% dos 505"] = (100 * uf["Todos os 505"] / len(perfil)).round(1)
display(uf.sort_values("Prioritários", ascending=False))
perfil.loc[prior, "Porte"].value_counts().to_frame("prioritários")

35 municípios no 4º quartil nos 6 rankings; 199 em alguns, mas não em todos
33 no 4º quartil nos 9 rankings que passam no filtro, 33 deles também na lista dos 6


,municípios,Água,Esgoto,Tratamento,Coleta,Destinação,zero_em_Tratamento,zero_em_Destinação,DRSAI_válidas,DRSAI_mediana
Grupo,,,,,,,,,,
Demais,470,71.1,37.04,26.10,82.64,52.44,0.30,0.45,333,49.39
Prioritários,35,49.1,18.12,6.52,68.05,0.00,0.43,1.00,27,79.71


,Prioritários,Todos os 505,% dos prioritários,% dos 505
UF,,,,
CE,20,95,57.1,18.8
BA,8,151,22.9,29.9
PB,4,65,11.4,12.9
MA,2,11,5.7,2.2
PI,1,6,2.9,1.2
AL,0,22,0.0,4.4
PE,0,77,0.0,15.2
RN,0,55,0.0,10.9
SE,0,23,0.0,4.6


,prioritários
Porte,
Pequeno e médio porte,35


**Os municípios prioritários**, do pior para o melhor pela posição mediana nos 6 rankings.

In [16]:
perfil["Posição mediana (6)"] = pos.loc[:, DIREITO].median(axis=1)
cols = ["Município", "UF", "Porte"] + mcda.NOMES + ["Posição ABES", "Posição mediana (6)", "DRSAI"]
perfil.loc[prior, cols].sort_values("Posição mediana (6)", ascending=False).round(1).reset_index(drop=True)

,Município,UF,Porte,Água,Esgoto,Tratamento,Coleta,Destinação,Posição ABES,Posição mediana (6),DRSAI
0,Aratuba,CE,Pequeno e médio porte,23.8,8.5,26.8,39.3,0.0,505,503.5,61.1
1,Baturité,CE,Pequeno e médio porte,62.0,1.8,1.6,43.3,0.0,502,502.0,208.2
2,Beberibe,CE,Pequeno e médio porte,16.6,9.4,31.0,41.8,0.0,504,502.0,59.3
3,Tanhaçu,BA,Pequeno e médio porte,45.8,25.3,0.0,44.2,0.0,500,500.0,18.4
4,Itarema,CE,Pequeno e médio porte,54.2,4.4,8.9,33.0,0.0,503,500.0,19.8
5,Cajazeirinhas,PB,Pequeno e médio porte,40.3,10.8,15.8,48.4,0.0,501,499.5,NaN
6,Jaguaretama,CE,Pequeno e médio porte,61.0,1.6,1.7,51.8,0.0,499,499.0,51.1
7,Cascavel,CE,Pequeno e médio porte,46.7,3.4,4.4,70.7,0.0,498,490.5,48.4
8,Nova Ibiá,BA,Pequeno e médio porte,48.8,37.2,0.0,52.2,0.0,490,489.5,NaN
9,Pedrão,BA,Pequeno e médio porte,32.0,3.5,0.0,100.0,0.0,494,484.0,NaN
